# IBM ART — MiniLM Cross-Encoder Reranker Security Test

**Model:** `cross-encoder/ms-marco-MiniLM-L6-v2` (22.7M parameters). This notebook evaluates pairwise ranking robustness. It does not claim to test discrete natural-language token edits: ART attacks operate on the continuous input word-embedding tensor supplied to the real reranker. This is a latent-space threat model. The measured security outcome is whether a relevant candidate loses its ranking against a fixed irrelevant candidate.

All attacks below are official IBM ART implementations. Each attempt is recorded as EXECUTED or FAILED; no fallback attack is substituted. Run on Google Colab; GPU recommended.

In [ ]:
!pip -q install adversarial-robustness-toolbox sentence-transformers transformers accelerate
import json, time, html, traceback, hashlib, platform
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent, BasicIterativeMethod, NewtonFool, SquareAttack, ZooAttack
from google.colab import files

MODEL_ID = 'cross-encoder/ms-marco-MiniLM-L6-v2'
MAX_LENGTH = 64
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT = Path('/content/minilm_reranker_art_results'); OUT.mkdir(exist_ok=True)
REPORT = OUT / 'minilm_reranker_art_report.html'
LOG = OUT / 'minilm_reranker_art.log'
log_lines = []
def log(msg):
    print(msg); log_lines.append(str(msg))

log(f'UTC={datetime.now(timezone.utc).isoformat()}')
log(f'Python={platform.python_version()} torch={torch.__version__} device={DEVICE}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).to(DEVICE).eval()
params = sum(p.numel() for p in model.parameters())
log(f'MODEL_LOADED={MODEL_ID}; parameters={params:,}; config={model.config.model_type}')

# Synthetic, non-sensitive ranking example. The relevant passage should outrank the irrelevant one.
query = 'How can a web application prevent SQL injection attacks?'
relevant = 'Use parameterized queries or prepared statements so user input is treated as data, not executable SQL. Apply least-privilege database accounts and validate inputs.'
irrelevant = 'The solar system contains eight planets. Jupiter is the largest planet and has many moons.'

def encode_pair(q, d):
    return tokenizer(q, d, padding='max_length', truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
fixed = encode_pair(query, irrelevant)
attacked = encode_pair(query, relevant)
with torch.no_grad():
    fixed_score = float(model(**{k:v.to(DEVICE) for k,v in fixed.items()}).logits.reshape(-1)[0].cpu())
    ids = attacked['input_ids'].to(DEVICE)
    mask = attacked['attention_mask'].to(DEVICE)
    token_types = attacked.get('token_type_ids', torch.zeros_like(attacked['input_ids'])).to(DEVICE)
    input_embeds = model.get_input_embeddings()(ids).detach()
    baseline_score = float(model(input_ids=ids, attention_mask=mask, token_type_ids=token_types).logits.reshape(-1)[0].cpu())
baseline_margin = baseline_score - fixed_score
if baseline_margin <= 0:
    raise RuntimeError(f'Synthetic baseline is not correctly ordered for this model/config: relevant={baseline_score:.6f}, irrelevant={fixed_score:.6f}. Choose a validated pair before attack testing.')
log(f'BASELINE relevant_score={baseline_score:.6f}; irrelevant_score={fixed_score:.6f}; margin={baseline_margin:.6f}')

# ART classifier adapter: class 0 means candidate A outranks fixed candidate B;
# class 1 means the pairwise order is reversed. This is an explicit ranking proxy.
class PairwiseRankingProxy(torch.nn.Module):
    def __init__(self, reranker, attention_mask, token_type_ids, competitor_score):
        super().__init__(); self.reranker = reranker
        self.register_buffer('attention_mask', attention_mask)
        self.register_buffer('token_type_ids', token_type_ids)
        self.competitor_score = float(competitor_score)
    def forward(self, candidate_embeds):
        mask = self.attention_mask.expand(candidate_embeds.shape[0], -1)
        token_types = self.token_type_ids.expand(candidate_embeds.shape[0], -1)
        outputs = self.reranker(inputs_embeds=candidate_embeds, attention_mask=mask, token_type_ids=token_types)
        margin = outputs.logits.reshape(-1) - self.competitor_score
        return torch.stack((margin, -margin), dim=1)

wrapped = PairwiseRankingProxy(model, mask, token_types, fixed_score).to(DEVICE).eval()
loss = torch.nn.CrossEntropyLoss()
classifier = PyTorchClassifier(model=wrapped, loss=loss, input_shape=tuple(input_embeds.shape[1:]), nb_classes=2,
    optimizer=torch.optim.Adam(wrapped.parameters(), lr=1e-6), device_type='gpu' if DEVICE.type=='cuda' else 'cpu')
x = input_embeds.detach().cpu().numpy().astype(np.float32)
y = np.array([0], dtype=np.int64)
pred = classifier.predict(x)
if int(np.argmax(pred[0])) != 0:
    raise RuntimeError('ART adapter smoke test failed: expected baseline pairwise class 0.')
log(f'ART_SMOKE_TEST=PASS; input_shape={x.shape}; prediction={pred.tolist()}')

# Conservative latent-embedding perturbation budgets. These are not token-space epsilon values.
attack_specs = [
    ('FastGradientMethod', lambda: FastGradientMethod(estimator=classifier, eps=0.01, norm=np.inf)),
    ('ProjectedGradientDescent', lambda: ProjectedGradientDescent(estimator=classifier, eps=0.01, eps_step=0.002, max_iter=5, norm=np.inf, verbose=False)),
    ('BasicIterativeMethod', lambda: BasicIterativeMethod(estimator=classifier, eps=0.01, eps_step=0.002, max_iter=5, verbose=False)),
    ('NewtonFool', lambda: NewtonFool(classifier=classifier, max_iter=5, eta=0.01, verbose=False)),
    ('SquareAttack', lambda: SquareAttack(estimator=classifier, eps=0.01, max_iter=10, p_init=0.1, batch_size=1, verbose=False)),
    ('ZooAttack', lambda: ZooAttack(classifier=classifier, max_iter=5, binary_search_steps=1, initial_const=0.001, nb_parallel=4, batch_size=1, verbose=False)),
]
results=[]
for name, factory in attack_specs:
    started=time.time(); row={'attack':name,'status':'ATTEMPTED','baseline_margin':baseline_margin}
    try:
        attack=factory()
        adversarial=attack.generate(x=x, y=y)
        adv_t=torch.as_tensor(adversarial, dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            adv_score=float(model(inputs_embeds=adv_t, attention_mask=mask, token_type_ids=token_types).logits.reshape(-1)[0].cpu())
        adv_margin=adv_score-fixed_score
        row.update(status='EXECUTED', adversarial_score=adv_score, fixed_competitor_score=fixed_score,
            adversarial_margin=adv_margin, ranking_inverted=bool(adv_margin <= 0),
            linf=float(np.max(np.abs(adversarial-x))), l2=float(np.linalg.norm((adversarial-x).reshape(-1))),
            duration_s=round(time.time()-started,3), interpretation='Pairwise rank inversion' if adv_margin <= 0 else 'No pairwise rank inversion in this run')
    except Exception as exc:
        row.update(status='FAILED', error=repr(exc), traceback=traceback.format_exc(), duration_s=round(time.time()-started,3))
    results.append(row); log(json.dumps(row, ensure_ascii=False))

summary={'utc':datetime.now(timezone.utc).isoformat(),'model':MODEL_ID,'parameters':params,'device':str(DEVICE),
 'input_type':'continuous word-embedding tensor for one query-document pair','objective':'relevant candidate score minus fixed irrelevant candidate score',
 'baseline_relevant_score':baseline_score,'baseline_irrelevant_score':fixed_score,'baseline_margin':baseline_margin,
 'attack_count':len(results),'executed':sum(r['status']=='EXECUTED' for r in results),'failed':sum(r['status']=='FAILED' for r in results),
 'rank_inversions':sum(r.get('ranking_inverted',False) for r in results),
 'limitations':'Latent word-embedding perturbations only. This does not produce valid modified text tokens and does not test tokenizer-level semantic-preserving edits, corpus poisoning, API extraction, or production retrieval integration.'}
log('SUMMARY '+json.dumps(summary, indent=2))

cols=['attack','status','baseline_margin','adversarial_margin','ranking_inverted','linf','l2','duration_s','interpretation','error']
rows=''.join('<tr>'+''.join('<td>'+html.escape(str(r.get(c,'')))+'</td>' for c in cols)+'</tr>' for r in results)
header=''.join('<th>'+html.escape(c)+'</th>' for c in cols)
report=f'''<!doctype html><html><head><meta charset="utf-8"><title>MiniLM Reranker IBM ART Report</title><style>body{{font:14px/1.5 Arial;margin:0;color:#172033}}header{{background:#172b4d;color:white;padding:24px}}main{{max-width:1300px;margin:24px auto;padding:0 18px}}.note{{background:#eef6ff;border-left:4px solid #2766a8;padding:14px;margin:18px 0}}table{{border-collapse:collapse;width:100%;font-size:12px}}th,td{{border:1px solid #d7dee8;padding:8px;vertical-align:top;overflow-wrap:anywhere}}th{{background:#eaf0f7}}pre{{white-space:pre-wrap;overflow-wrap:anywhere;background:#f6f8fb;padding:12px}}</style></head><body><header><h1>MiniLM Cross-Encoder Reranker — IBM ART Report</h1><div>{html.escape(MODEL_ID)} · actual PyTorch model · ART attacks</div><small>{html.escape(summary['utc'])}</small></header><main><div class="note"><b>Threat-model scope:</b> ART perturbs continuous input word embeddings for the relevant query-document pair while a competitor passage remains fixed. Success means the relevant passage score falls to or below the competitor score. This is a latent-space ranking proxy, not a discrete text/token attack. Validate with token-level attacks and a real retrieval corpus before treating results as production impact.</div><h2>Run summary</h2><pre>{html.escape(json.dumps(summary,indent=2))}</pre><h2>Attack results</h2><table><thead><tr>{header}</tr></thead><tbody>{rows}</tbody></table><h2>Test input</h2><pre>Query: {html.escape(query)}\nRelevant passage: {html.escape(relevant)}\nFixed competitor: {html.escape(irrelevant)}</pre><h2>Full results JSON</h2><pre>{html.escape(json.dumps(results,indent=2))}</pre></main></body></html>'''
REPORT.write_text(report,encoding='utf-8')
LOG.write_text('\n'.join(log_lines)+'\n',encoding='utf-8')
print('HTML report:',REPORT,'bytes=',REPORT.stat().st_size)
print('Log:',LOG,'bytes=',LOG.stat().st_size)
files.download(str(REPORT)); files.download(str(LOG))


## Interpretation rules

- `EXECUTED` means ART generated an adversarial embedding and the reranker was evaluated on it.
- `FAILED` means no valid adversarial result was produced; the exception is retained.
- `ranking_inverted=True` is a **potential ranking robustness finding**, not automatically a confirmed security vulnerability.
- The perturbation is applied to continuous word embeddings. It cannot be represented directly as readable text without a separate discrete-token reconstruction/validation stage.
- The sample uses one synthetic query and two passages. A defensible assessment requires many held-out query/document pairs and downstream retrieval evaluation.